In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

df = spark.readStream.format("delta").table("sales_project_streaming.brz.employees")

schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("before", StructType([
        StructField("employee_name", StringType(), True),
        StructField("role", StringType(), True),
        StructField("department", StringType(), True),
        StructField("region_id", IntegerType(), True),
        StructField("joining_date", DateType(), True),
        StructField("salary", LongType(), True)
    ]), True),
    StructField("after", StructType([
        StructField("employee_name", StringType(), True),
        StructField("role", StringType(), True),
        StructField("department", StringType(), True),
        StructField("region_id", IntegerType(), True),
        StructField("joining_date", DateType(), True),
        StructField("salary", LongType(), True)
    ]), True),
    StructField("operation", StringType(), True),
    StructField("event_time", TimestampType(), True)
])

parsed_emp = df.withColumn("parsed", F.from_json(F.col("value"), schema)).select("parsed", "ingestion_ts")

emp_df = parsed_emp.select("parsed.employee_id", F.col("parsed.before.employee_name").alias("before_employee_name"),
                   F.col("parsed.before.role").alias("before_role"),
                   F.col("parsed.before.department").alias("before_department"),
                   F.col("parsed.before.region_id").alias("before_region_id"),
                   F.col("parsed.before.joining_date").alias("before_joining_date"),
                   F.col("parsed.before.salary").alias("before_salary"),
                   F.col("parsed.after.employee_name").alias("after_employee_name"),
                   F.col("parsed.after.role").alias("after_role"),
                   F.col("parsed.after.department").alias("after_department"),
                   F.col("parsed.after.region_id").alias("after_region_id"),
                   F.col("parsed.after.joining_date").alias("after_joining_date"),
                   F.col("parsed.after.salary").alias("after_salary"),
                   "parsed.operation", "parsed.event_time", F.current_timestamp().alias("processed_time"))

w = Window.partitionBy("employee_id", "operation").orderBy(F.col("event_time").desc())

# emp_df = ()

def scd_imp(batch_df, batch_id):

    batch_df = (batch_df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn")==1)
          .drop("rn"))

    batch_df.createOrReplaceTempView("source_view")

    spark.sql("""merge into sales_project_streaming.slv.employees t
              using source_view s
              on s.employee_id = t.employee_id and t.is_current = True

                --- if Deleted - Expire current record and set is_deleted to True

              when matched and s.operation = "D"
              then update set t.end_date = to_date(s.event_time),
                                t.is_current = False,
                                t.is_deleted = True,
                                t.processed_time = s.processed_time

                --- if updated - Expire current record and set is_current to False

               when matched and s.operation = "U"
               then update set t.end_date = to_date(s.event_time),
                                t.is_current = False,
                                t.processed_time = s.processed_time;""")
                
                # Insert DF for updates and inserts
    
    insert_df = (batch_df.filter(F.col("operation").isin(["I","U"]))
                 .select("employee_id", F.col("after_employee_name").alias("employee_name"),
                         F.col("after_role").alias("role"), F.col("after_department").alias("department"),
                         F.col("after_region_id").alias("region_id"), F.col("after_joining_date").alias("joining_date"),
                         F.col("after_salary").alias("salary"), F.to_date(F.col("event_time")).alias("start_date"),
                         F.lit(None).alias("end_date"), F.lit(True).alias("is_current"),
                         F.lit(False).alias("is_deleted"), "processed_time"))
    
    insert_df.write.format("delta").mode("append").saveAsTable("sales_project_streaming.slv.employees")

query = (emp_df.writeStream.foreachBatch(scd_imp)
         .option("checkpointLocation", "/Volumes/sales_project_streaming/slv/checkpoints_vol/employees_chck_slv/")
         .trigger(availableNow = True)
         .start())

query.awaitTermination()